## Project description

<span style="color:white; background:#E8A33D ; padding:4px 10px">
  Smart Light Controller — Dimmer
</span>

### Goal Based Agent

The agents in 01–03 all had their rule written into the code. "If it's below
400, turn the light on." Change what you want and you have to change the `if`.

A goal-based agent is given a **target** and works out the action itself.
The goal is data, not code.

**The change:** the light is no longer on or off. It's a dimmer, 0 to 100%.
The goal is a target light level in the room — say 500.

The room's total light is daylight plus whatever the lamp adds. So the agent
asks: how much light is missing, and how bright does the lamp need to be to
cover the gap?

Same agent, different goal number, different behaviour. No code changes.

In [6]:
# Class for a goal-based agent that adjusts lamp output to reach a target light level
class GoalBasedAgent:
    def __init__(self, target_level=550, lamp_max_output=1000):
        self.target_level = target_level
        self.lamp_max_output = lamp_max_output
        self.brightness = 0   # current dimmer setting, 0-100

# Decide method to determine the lamp output based on the current daylight level
    def decide(self, daylight):
        gap = self.target_level - daylight

        if gap <= 0:
            return 0

        needed = (gap / self.lamp_max_output) * 100
        return min(100, round(needed))

# Act method to adjust the lamp output based on the decision made
    def act(self, daylight):
        new_brightness = self.decide(daylight)
        changed = new_brightness != self.brightness
        self.brightness = new_brightness
        return changed

In [7]:
# Main execution block to simulate the agent's behavior with varying daylight readings
daylight_readings = [50, 150, 300, 450, 600, 750, 600, 450, 300, 150, 50]

agent = GoalBasedAgent(target_level=550)

print(f"{'Daylight':>10} {'Lamp %':>8} {'Room total':>12}")
for daylight in daylight_readings:
    agent.act(daylight)
    lamp_contribution = (agent.brightness / 100) * agent.lamp_max_output
    total = daylight + lamp_contribution
    print(f"{daylight:>10} {agent.brightness:>8} {total:>12.0f}")

  Daylight   Lamp %   Room total
        50       50          550
       150       40          550
       300       25          550
       450       10          550
       600        0          600
       750        0          750
       600        0          600
       450       10          550
       300       25          550
       150       40          550
        50       50          550


### Same code, different goal

Nothing below changes the class. Only the number passed in.

In [8]:
# Test the agent with different target levels
for target in (300, 550, 800, 1000):
    agent = GoalBasedAgent(target_level=target)
    settings = []
    for daylight in daylight_readings:
        agent.act(daylight)
        settings.append(agent.brightness)
    print(f"target {target}: {settings}")

target 300: [25, 15, 0, 0, 0, 0, 0, 0, 0, 15, 25]
target 550: [50, 40, 25, 10, 0, 0, 0, 10, 25, 40, 50]
target 800: [75, 65, 50, 35, 20, 5, 20, 35, 50, 65, 75]
target 1000: [95, 85, 70, 55, 40, 25, 40, 55, 70, 85, 95]


## Conclusions

In 01–03 the rule lived in the code. "If below 400, turn on." To change what
the agent does, I had to edit an `if`.

Here the agent gets a **target number** and works out the rest by itself.

### How it works

The lamp is a dimmer, 0 to 100%. The room's light is daylight plus whatever
the lamp adds.

The agent asks one question: how far is the room from the target?
Then it sets the dimmer to cover that gap.

    gap = target - daylight
    brightness = (gap / lamp_max_output) * 100

No thresholds. Just arithmetic.

### One goal, one day

| Daylight | Lamp % | Room total |
| -------- | ------ | ---------- |
| 50 | 50 | 550 |
| 150 | 40 | 550 |
| 300 | 25 | 550 |
| 450 | 10 | 550 |
| 600 | 0 | 600 |
| 750 | 0 | 750 |

The room stays at 550 all morning. The lamp fades out as daylight rises, on
its own.

At midday it goes to 0 and the room goes over target — 600, then 750. The agent
can't do anything about that. It has a lamp, not a blind. It can add light but
never remove it, so the goal is only reachable from below.

### Same code, four goals

    target 300:  [25, 15, 0, 0, 0, 0, 0, 0, 0, 15, 25]
    target 550:  [50, 40, 25, 10, 0, 0, 0, 10, 25, 40, 50]
    target 800:  [75, 65, 50, 35, 20, 5, 20, 35, 50, 65, 75]
    target 1000: [95, 85, 70, 55, 40, 25, 40, 55, 70, 85, 95]

Four different behaviours. The class was not touched once — only the number
passed to `GoalBasedAgent()`.

At target 300 the lamp sleeps through the middle of the day. At target 1000 it
never switches off at all, not even at noon.

### Compared to the earlier agents

The model-based agent in 03 had memory. Give it the same reading twice and it
could answer differently, depending on what came before.

This agent has no memory. The same daylight always gives the same brightness.
That's why all four rows above are symmetrical.

So it's not simply "smarter" than 03. It dropped memory and gained a goal.
Different tool, different job.

### What I learned

- Putting a target in `__init__` instead of hardcoding a threshold
- Working out an action from a gap rather than a rule
- `min(100, ...)` to keep the dimmer inside its real limits
- An agent can have a goal it cannot reach, and it should still do what it can
- Naming things consistently — I called it `current_lamp_output` in `__init__`
  and `self.brightness` in `act()`, and it broke